In [206]:
import torch as t 
from torch import nn
import pandas as pd 
from pprint import pprint


df = pd.read_json("data.json")

x = df["input"]

tokenized = []
for sentence in x : 
    tokenized.append(sentence.split())
    
    
flatten = [word.lower() for sentence in tokenized for word in sentence]
for word in df['target']:
    flatten.append(word)


vocab = {word:idx for idx,word in enumerate(set(flatten))}
print (vocab)
# turn tokenized to ids
final_x = []
for sentence in tokenized: 
    hold = []
    for word in sentence : 
        if word in vocab.keys():
            hold.append(vocab[word])
    final_x.append(t.tensor(hold))


from torch.nn.utils.rnn import pad_sequence

vocab_size = len(vocab)
batch_input = pad_sequence(final_x , batch_first=True , padding_side="right")

batch_input

# target_indices لكل جملة
# target_indices = t.tensor([sample["target_index"] for sample in df])
indices = []
for index in df["target_index"]:
    indices.append(t.tensor(index))
    
    
target_words = t.tensor([vocab[sample] for sample in df['target']])



pprint((batch_input , indices ,target_words))


{'buy': 0, 'went': 1, 'roof': 2, 'sets': 3, 'milk.': 4, 'book': 5, 'a': 6, 'football': 7, 'to': 8, 'he': 9, 'she': 10, 'last': 11, 'play': 12, 'store': 13, 'day.': 14, 'every': 15, 'market': 16, 'they': 17, 'an': 18, 'sunday.': 19, 'in': 20, 'sun': 21, 'going': 22, 'sat': 23, 'ate': 24, '[mask]': 25, 'farm.': 26, 'night.': 27, 'read': 28, 'bed.': 29, 'the': 30, 'before': 31, 'yesterday': 32, 'apple': 33, 'at': 34, 'on': 35, 'ahmed': 36, 'cat': 37, 'yesterday.': 38, 'west': 39, 'ali': 40}
(tensor([[18, 33, 32, 34, 30, 26],
        [ 1,  8, 30, 38,  0,  0],
        [ 7, 15,  0,  0,  0,  0],
        [37, 35, 30,  2, 11, 27],
        [ 6,  5, 31, 22,  8, 29],
        [ 8, 30, 13,  8,  0,  4],
        [21, 20, 30, 39, 15, 14]]),
 [tensor(1), tensor(4), tensor(1), tensor(2), tensor(1), tensor(1), tensor(2)],
 tensor([24, 16, 12, 23, 28,  1,  3]))


In [207]:
class Model(nn.Module):
    def __init__(self , vocab_size , embedding_size , hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size , embedding_size)
        self.rnn = nn.LSTM(embedding_size , hidden_size , bidirectional=True)
        self.linear = nn.Linear(hidden_size*2 , vocab_size)
        
    def forward(self , x , indices):
        emb = self.embedding(x)
        out , _ = self.rnn(emb)
        mask_hidden = out[t.arange(x.size(0)),indices, : ]
        out = self.linear(mask_hidden)
        return out 

In [ ]:

embedding_dim = 8
hidden_dim = 16
learning_rate = 0.01
num_epochs = 1000


model = Model(vocab_size=len(vocab), embedding_size=embedding_dim, hidden_size=hidden_dim)
optimizer = t.optim.Adam(model.parameters(), lr=learning_rate)

import torch.nn.functional as F

for epoch in range(1, num_epochs+1):
    model.train()
    optimizer.zero_grad()
    logits = model(batch_input, indices)
    loss = F.cross_entropy(logits, target_words)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


model.eval()
with t.no_grad():
    logits = model(batch_input, indices)
    predicted_ids = t.argmax(logits, dim=1)


id_to_word = {id_: word for word, id_ in vocab.items()}
predicted_words = [id_to_word[i.item()] for i in predicted_ids]

print("\nPredicted words:", predicted_words)
print("Target words:", [id_to_word[i.item()] for i in target_words])

Epoch 50, Loss: 0.0343
Epoch 100, Loss: 0.0063
Epoch 150, Loss: 0.0035
Epoch 200, Loss: 0.0022
Epoch 250, Loss: 0.0016
Epoch 300, Loss: 0.0012
Epoch 350, Loss: 0.0009
Epoch 400, Loss: 0.0007
Epoch 450, Loss: 0.0006
Epoch 500, Loss: 0.0005
Epoch 550, Loss: 0.0004
Epoch 600, Loss: 0.0004
Epoch 650, Loss: 0.0003
Epoch 700, Loss: 0.0003
Epoch 750, Loss: 0.0003
Epoch 800, Loss: 0.0002
Epoch 850, Loss: 0.0002
Epoch 900, Loss: 0.0002
Epoch 950, Loss: 0.0002
Epoch 1000, Loss: 0.0002

Predicted words: ['ate', 'market', 'play', 'sat', 'read', 'went', 'sets']
Target words: ['ate', 'market', 'play', 'sat', 'read', 'went', 'sets']


In [ ]:
import torch as t

# جملة جديدة (توكنز + [MASK])
new_sentence = ["Ahmed", "[MASK]", "an", "apple", "yesterday", "at", "the", "farm."]
new_target_index = 1  # مكان [MASK]


new_input_ids = t.tensor([[vocab[word.lower()] for word in new_sentence]], dtype=t.long)  

# Prediction
model.eval()
with t.no_grad():
    logits = model(new_input_ids, t.tensor([new_target_index]))
    probs = t.softmax(logits, dim=1)         
    predicted_id = t.argmax(probs, dim=1).item()

predicted_word = [word for word, id_ in vocab.items() if id_ == predicted_id][0]

print("Predicted word at [MASK]:", predicted_word)
print("Probabilities:", probs)

Predicted word at [MASK]: ate
Probabilities: tensor([[3.4929e-04, 5.2579e-02, 2.5231e-04, 2.6381e-02, 3.3799e-04, 3.3056e-04,
         2.4130e-04, 3.5081e-04, 4.2886e-04, 5.0139e-04, 3.1557e-04, 3.4593e-04,
         2.8774e-02, 3.9285e-04, 3.6787e-04, 4.2888e-04, 9.6606e-03, 4.9665e-04,
         2.2426e-04, 4.5733e-04, 3.1933e-04, 2.0768e-04, 3.2792e-04, 4.5742e-03,
         8.6263e-01, 4.4560e-04, 3.7251e-04, 4.1988e-04, 3.2592e-03, 3.9353e-04,
         3.1599e-04, 4.0570e-04, 3.7602e-04, 2.7182e-04, 3.4329e-04, 4.6718e-04,
         3.0350e-04, 2.0193e-04, 2.2688e-04, 4.5745e-04, 4.5997e-04]])
